# Deep Learning - FSL assignment

The goal of this assignment is training a prototypical neural network with the Omniglot dataset using episodic learning. Once trained, we will evaluate its accuracy on the test set.

We begin declaring the required libraries and setting the hyperparameters

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import Omniglot
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

# --------------------------
# Hyperparameters and settings
# --------------------------
k_way = 3
n_support = 5
n_query = 15

episodes_per_epoch = 200   
num_epochs = 20
learning_rate = 0.001
weight_decay = 1e-4        # NEW

data_root = './data'
test_episodes = 600


In [10]:
import numpy as np

def set_seed(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(0)

The [Omniglot data set](https://github.com/brendenlake/omniglot) contains 50 alphabets. We split these into a background set of 30 alphabets and an evaluation set of 20 alphabets. Each of the 1623 characters was drawn online via Amazon's Mechanical Turk by 20 different people.

We prepare the data for episodic learning

In [3]:
# --------------------------
# Episodic Dataset for Omniglot (with rotation-as-classes augmentation for training)
# --------------------------
class OmniglotEpisodicDataset(Dataset):
    """
    Creates episodes for few-shot learning from Omniglot.
    Optional augmentation: treat rotations {0,90,180,270} as different classes (training only).
    """
    ROTATIONS = (0, 90, 180, 270)

    def __init__(self, root, transform, k_way, n_support, n_query, background=True, rotations_as_classes=False):
        self.dataset = Omniglot(root=root, background=background, download=True, transform=transform)
        self.k_way = k_way
        self.n_support = n_support
        self.n_query = n_query
        self.rotations_as_classes = rotations_as_classes

        # Build a dictionary mapping each base class to list of indices
        self.base_classes = {}
        for idx, (_, label) in enumerate(self.dataset):
            if label not in self.base_classes:
                self.base_classes[label] = []
            self.base_classes[label].append(idx)

        self.base_keys = list(self.base_classes.keys())

        # If enabled, expand keys into (base_label, rotation) pairs
        if self.rotations_as_classes:
            self.keys = [(lbl, rot) for lbl in self.base_keys for rot in self.ROTATIONS]
        else:
            self.keys = self.base_keys

    def __len__(self):
        return 100000

    def __getitem__(self, index):
        selected_classes = random.sample(self.keys, self.k_way)

        support_images, support_labels = [], []
        query_images, query_labels = [], []

        for new_label, cls in enumerate(selected_classes):
            if self.rotations_as_classes:
                base_label, rot = cls
            else:
                base_label, rot = cls, 0

            indices = self.base_classes[base_label]
            selected_indices = random.sample(indices, self.n_support + self.n_query)
            support_idx = selected_indices[:self.n_support]
            query_idx = selected_indices[self.n_support:]

            for idx_ in support_idx:
                img, _ = self.dataset[idx_]  # tensor after transform: [C,H,W]
                if rot != 0:
                    img = TF.rotate(img, rot, interpolation=InterpolationMode.NEAREST)
                support_images.append(img)
                support_labels.append(new_label)

            for idx_ in query_idx:
                img, _ = self.dataset[idx_]
                if rot != 0:
                    img = TF.rotate(img, rot, interpolation=InterpolationMode.NEAREST)
                query_images.append(img)
                query_labels.append(new_label)

        support_images = torch.stack(support_images, dim=0)
        support_labels = torch.tensor(support_labels, dtype=torch.long)
        query_images = torch.stack(query_images, dim=0)
        query_labels = torch.tensor(query_labels, dtype=torch.long)

        return support_images, support_labels, query_images, query_labels

Model definition. A simple CNN with 4 blocks is used.

In [4]:
# --------------------------
# Prototypical Network Model
# --------------------------
class ProtoNet(nn.Module):
    """
    A simple convolutional encoder that maps images into an embedding space.
    Typically, a 4-block ConvNet is used for Omniglot.
    """
    def __init__(self, x_dim=1, hid_dim=64, z_dim=64):
        """
        x_dim: number of input channels (1 for grayscale)
        hid_dim: number of hidden channels
        z_dim: dimension of the final embedding
        """
        super(ProtoNet, self).__init__()
        self.encoder = nn.Sequential(
            # Block 1
            nn.Conv2d(x_dim, hid_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(hid_dim),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 2
            nn.Conv2d(hid_dim, hid_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(hid_dim),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 3
            nn.Conv2d(hid_dim, hid_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(hid_dim),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 4
            nn.Conv2d(hid_dim, z_dim, kernel_size=3, padding=1),
            nn.BatchNorm2d(z_dim),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

    def forward(self, x):
        """
        Forward pass: input x shape: [batch, channels, height, width]
        Returns: embeddings of shape [batch, z_dim]
        """
        out = self.encoder(x)
        return out.view(out.size(0), -1)


Training function

In [5]:
# --------------------------
# Training Function for one Epoch
# --------------------------
def train(model, optimizer, dataloader, device):
    model.train()
    total_loss = 0.0

    for batch_idx, (support_images, support_labels, query_images, query_labels) in enumerate(dataloader):
        support_images = support_images.squeeze(0).to(device)
        support_labels = support_labels.squeeze(0).to(device)
        query_images = query_images.squeeze(0).to(device)
        query_labels = query_labels.squeeze(0).to(device)

        optimizer.zero_grad()

        embeddings_support = model(support_images)   # [k_way*n_support, z_dim]
        embeddings_query = model(query_images)       # [k_way*n_query, z_dim]

        embedding_dim = embeddings_support.size(-1)
        embeddings_support = embeddings_support.view(k_way, n_support, embedding_dim)
        prototypes = embeddings_support.mean(dim=1)  # [k_way, z_dim]

        distances = (embeddings_query.unsqueeze(1) - prototypes.unsqueeze(0)).pow(2).sum(dim=2)  # [k_way*n_query, k_way]
        logits = -distances

        loss = F.cross_entropy(logits, query_labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (batch_idx + 1) % 10 == 0:
            print(f"Episode {batch_idx+1}/{episodes_per_epoch}, Loss: {loss.item():.4f}")

        if batch_idx + 1 >= episodes_per_epoch:
            break

    return total_loss / episodes_per_epoch

Let's run training and check that it works properly.

In [6]:
# Set device: use GPU if available.


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Deterministic preprocessing (augmentation handled inside the dataset for training)
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Lambda(TF.invert),           # NEW: invert (common for Omniglot)
    transforms.Normalize((0.5,), (0.5,))
])



# Create the episodic training dataset (use background set + rotation-as-classes)
train_dataset = OmniglotEpisodicDataset(
    root=data_root,
    transform=transform,
    k_way=k_way,
    n_support=n_support,
    n_query=n_query,
    background=True,
    rotations_as_classes=True                      # NEW
)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

weight_decay = 1e-4     

# Initialize the model and optimizer (+ weight decay) and scheduler
model = ProtoNet(x_dim=1, hid_dim=64, z_dim=64).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# Training loop.
for epoch in range(1, num_epochs + 1):
    avg_loss = train(model, optimizer, train_loader, device)
    scheduler.step()
    print(f"Epoch {epoch}/{num_epochs}, Average Loss: {avg_loss:.4f}")

print("Training complete and model saved.")

Episode 10/200, Loss: 0.9014
Episode 20/200, Loss: 0.4347
Episode 30/200, Loss: 0.6185
Episode 40/200, Loss: 0.8392
Episode 50/200, Loss: 0.1340
Episode 60/200, Loss: 0.1660
Episode 70/200, Loss: 0.1172
Episode 80/200, Loss: 0.0975
Episode 90/200, Loss: 0.0193
Episode 100/200, Loss: 0.0797
Episode 110/200, Loss: 0.1501
Episode 120/200, Loss: 0.6650
Episode 130/200, Loss: 0.0158
Episode 140/200, Loss: 0.0801
Episode 150/200, Loss: 0.0452
Episode 160/200, Loss: 0.0587
Episode 170/200, Loss: 0.3806
Episode 180/200, Loss: 0.0028
Episode 190/200, Loss: 0.0773
Episode 200/200, Loss: 0.0638
Epoch 1/20, Average Loss: 0.1809
Episode 10/200, Loss: 0.0349
Episode 20/200, Loss: 0.2758
Episode 30/200, Loss: 0.0254
Episode 40/200, Loss: 0.0069
Episode 50/200, Loss: 0.6265
Episode 60/200, Loss: 0.0737
Episode 70/200, Loss: 0.0291
Episode 80/200, Loss: 0.1619
Episode 90/200, Loss: 0.0858
Episode 100/200, Loss: 0.0451
Episode 110/200, Loss: 0.0168
Episode 120/200, Loss: 0.0204
Episode 130/200, Loss: 0.

Once we have trained our model, we can evaluate it using the Omniglot test set. Since we only trained the embeddings, nearest neighbour is necessary to obtain the class.

In [7]:
# --------------------------
# Evaluation Function
# --------------------------
def evaluate(model, dataloader, device):
    """
    Evaluate the model over episodes and return the average query accuracy.
    """
    model.eval()
    total_acc = 0.0
    total_episodes = 0

    with torch.no_grad():
        for batch_idx, (support_images, support_labels, query_images, query_labels) in enumerate(dataloader):
            support_images = support_images.squeeze(0).to(device)  # shape: [k_way*n_support, C, H, W]
            support_labels = support_labels.squeeze(0).to(device)
            query_images = query_images.squeeze(0).to(device)      # shape: [k_way*n_query, C, H, W]
            query_labels = query_labels.squeeze(0).to(device)

            # Compute embeddings for support and query images.
            embeddings_support = model(support_images)            # shape: [k_way*n_support, z_dim]
            embeddings_query = model(query_images)                # shape: [k_way*n_query, z_dim]

            # Compute prototypes (mean embedding support) for each class.
            embedding_dim = embeddings_support.size(-1)
            embeddings_support = embeddings_support.view(k_way, n_support, embedding_dim)
            prototypes = embeddings_support.mean(dim=1)           # shape: [k_way, z_dim]

            # Compute distances and obtain predictions with softmax.
            distances = (embeddings_query.unsqueeze(1) - prototypes.unsqueeze(0)).pow(2).sum(dim=2)  # [k_way*n_query, k_way]
            logits = -distances
            pred_labels = logits.argmax(dim=1)

            # Calculate accuracy for this episode.
            acc = (pred_labels == query_labels).float().mean().item()
            total_acc += acc
            total_episodes += 1

            if total_episodes >= test_episodes:
                break

    avg_acc = total_acc / total_episodes
    return avg_acc

Get the accuracy results on the Omniglot evaluation set

In [8]:
# Create episodic test dataset using the evaluation set (background=False).
test_dataset = OmniglotEpisodicDataset(
    root=data_root,
    transform=transform,
    k_way=k_way,
    n_support=n_support,
    n_query=n_query,
    background=False,
    rotations_as_classes=False     # IMPORTANT: disable for evaluation
)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda")
)

test_acc = evaluate(model, test_loader, device)
print(f"\nTest Accuracy over {test_episodes} episodes: {test_acc*100:.2f}%")


Test Accuracy over 600 episodes: 98.78%


Answer the following questions:
* For the given dataset, why do you think that episodic learning outperforms standard training (i.e., with all the training set)?

Episodic learning outperforms standard training on Omniglot because it matches the few-shot test scenario: in each episode the model must build class prototypes from a small support set and classify query images by comparing distances in the embedding space. This directly optimizes the representation to make same-class samples close and different-class samples far apart under a fixed metric, which is exactly what is needed for unseen classes at inference time. In contrast, standard training with a fixed classifier over all training classes tends to learn features specialized to that closed label set, which transfers worse when the model must recognize new classes using only a handful of examples.



* Make experiments changing the values K=3 and K=10 and write the conclusions.